# 🍱 우리 학교 급식 데이터 AI 탐험대
## 나에게 맞는 급식 개인추천기 — 학생용 완성 프로젝트

이 노트북 하나만 Colab에 업로드하면 실행할 수 있습니다. 5회 동안 각 단계를 이해하고 바꾸면서 우리 팀의 서비스를 완성합니다.

### 완성 후 설명할 수 있어야 하는 것

1. NEIS API가 학교 급식 데이터를 주는 과정
2. 메뉴 문자열을 분석 가능한 표로 바꾸는 전처리
3. TF-IDF가 한국어 메뉴를 숫자로 바꾸는 방식
4. 코사인 유사도가 취향과 메뉴를 비교하는 방식
5. K-Means가 비슷한 식단을 묶는 방식
6. AI 추천이 정답이 아니라는 점과 알레르기 안전 원칙

> 개인정보 약속: 이름·학번·반·연락처·체중·질병명은 입력하지 않습니다. 코드 셀에는 가상 프로필만 둡니다. Colab 서비스는 임시 공개 링크이므로 실제 알레르기·질병 정보는 입력하지 않고 수업용 가상 번호만 사용합니다.


## 0단계: 실행 환경 확인

Colab에는 대부분의 라이브러리가 준비되어 있습니다. 서비스 화면용 Gradio가 없을 때만 설치합니다.

- 예상 결과: `환경 준비 완료`가 출력됩니다.
- 확인 질문: 라이브러리는 직접 모든 코드를 쓰지 않고도 검증된 기능을 사용할 수 있게 해 줍니다.


In [ ]:
import importlib.util
import os
import subprocess
import sys

VERIFY_MODE = os.getenv("NEIS_MEAL_AI_VERIFY", "0") == "1"
if not VERIFY_MODE and importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44,<7"], check=True)
print("환경 준비 완료", "(자동 검증 모드)" if VERIFY_MODE else "(Colab 학습 모드)")


## 1단계: NEIS API 이해하기

API는 다른 서비스가 공개한 정보를 정해진 주소와 규칙으로 요청하는 창구입니다. 아래 코드는 학교명으로 학교 코드를 찾고, 그 코드로 급식 행을 가져옵니다.

- 입력: 학교명, 시작일, 종료일
- 출력: 메뉴·열량·영양·알레르기 번호가 들어 있는 JSON 행
- 학생 도전: `pSize`를 찾아 한 번에 몇 행을 요청하는지 확인하세요.


In [ ]:
"""NEIS 교육정보 개방 포털의 학교와 급식 API 경계."""

from __future__ import annotations

import os
from dataclasses import dataclass
from datetime import date, datetime
from typing import Any, Callable

import requests


NEIS_BASE_URL = "https://open.neis.go.kr/hub"
NEIS_PAGE_SIZE = 1000
HttpGet = Callable[..., Any]


class NeisApiError(RuntimeError):
    """NEIS 조회 실패를 학생이 이해할 수 있는 한 종류의 오류로 표현한다."""


@dataclass(frozen=True)
class SchoolInfo:
    """급식 조회에 필요한 최소 학교 정보."""

    name: str
    office_code: str
    school_code: str
    school_kind: str = ""
    address: str = ""


def validate_date_range(start: str, end: str) -> tuple[date, date]:
    """YYYYMMDD 조회 범위를 검증하고 날짜 객체로 돌려준다."""

    try:
        start_date = datetime.strptime(start, "%Y%m%d").date()
        end_date = datetime.strptime(end, "%Y%m%d").date()
    except ValueError as exc:
        raise ValueError("날짜는 YYYYMMDD 형식의 실제 날짜여야 합니다.") from exc
    if start_date > end_date:
        raise ValueError("시작일은 종료일보다 늦을 수 없습니다.")
    if (end_date - start_date).days + 1 > 366:
        raise ValueError("조회 기간은 366일 이하여야 합니다.")
    return start_date, end_date


def _request_json(
    endpoint: str,
    params: dict[str, Any],
    *,
    http_get: HttpGet,
) -> dict[str, Any]:
    api_key = os.getenv("NEIS_API_KEY", "").strip()
    if api_key:
        params = {**params, "KEY": api_key}
    try:
        response = http_get(
            f"{NEIS_BASE_URL}/{endpoint}",
            params=params,
            timeout=15,
        )
        response.raise_for_status()
        payload = response.json()
    except Exception as exc:
        raise NeisApiError("NEIS 서버에 연결하지 못했습니다. 잠시 후 다시 시도하세요.") from exc
    if not isinstance(payload, dict):
        raise NeisApiError("NEIS 응답 형식을 해석할 수 없습니다.")
    return payload


def _extract_rows(payload: dict[str, Any], dataset_name: str) -> list[dict[str, Any]]:
    root_result = payload.get("RESULT")
    if isinstance(root_result, dict):
        if root_result.get("CODE") == "INFO-200":
            return []
        raise NeisApiError(f"NEIS 오류: {root_result.get('MESSAGE', '알 수 없는 오류')}")

    dataset = payload.get(dataset_name)
    if not isinstance(dataset, list) or len(dataset) < 2:
        raise NeisApiError("NEIS 응답 형식을 해석할 수 없습니다.")

    head = dataset[0].get("head", []) if isinstance(dataset[0], dict) else []
    result = next(
        (item.get("RESULT") for item in head if isinstance(item, dict) and "RESULT" in item),
        None,
    )
    if isinstance(result, dict) and result.get("CODE") not in (None, "INFO-000"):
        if result.get("CODE") == "INFO-200":
            return []
        raise NeisApiError(f"NEIS 오류: {result.get('MESSAGE', '알 수 없는 오류')}")

    rows = dataset[1].get("row") if isinstance(dataset[1], dict) else None
    if rows is None:
        return []
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        raise NeisApiError("NEIS 응답 형식을 해석할 수 없습니다.")
    return rows


def _extract_total_count(payload: dict[str, Any], dataset_name: str) -> int | None:
    """NEIS head의 전체 행 수를 읽는다. 값이 없으면 단일 페이지로 처리한다."""

    dataset = payload.get(dataset_name)
    if not isinstance(dataset, list) or not dataset:
        return None
    head = dataset[0].get("head", []) if isinstance(dataset[0], dict) else []
    item = next(
        (part for part in head if isinstance(part, dict) and "list_total_count" in part),
        None,
    )
    if item is None:
        return None
    try:
        total = int(item["list_total_count"])
    except (TypeError, ValueError) as exc:
        raise NeisApiError("NEIS 전체 데이터 개수를 해석할 수 없습니다.") from exc
    return max(total, 0)


def search_school(name: str, *, http_get: HttpGet = requests.get) -> SchoolInfo:
    """정확한 학교명으로 교육청 코드와 학교 코드를 찾는다."""

    cleaned_name = name.strip()
    if not cleaned_name:
        raise ValueError("학교명을 입력하세요.")
    payload = _request_json(
        "schoolInfo",
        {
            "Type": "json",
            "pIndex": 1,
            "pSize": 100,
            "SCHUL_NM": cleaned_name,
        },
        http_get=http_get,
    )
    rows = _extract_rows(payload, "schoolInfo")
    exact_rows = [row for row in rows if str(row.get("SCHUL_NM", "")).strip() == cleaned_name]
    if not exact_rows:
        raise NeisApiError(f"'{cleaned_name}'와 정확히 일치하는 학교를 찾지 못했습니다.")
    row = exact_rows[0]
    return SchoolInfo(
        name=str(row.get("SCHUL_NM", "")),
        office_code=str(row.get("ATPT_OFCDC_SC_CODE", "")),
        school_code=str(row.get("SD_SCHUL_CODE", "")),
        school_kind=str(row.get("SCHUL_KND_SC_NM", "")),
        address=str(row.get("ORG_RDNMA", "")),
    )


def fetch_meals(
    school: SchoolInfo,
    start: str,
    end: str,
    *,
    http_get: HttpGet = requests.get,
) -> list[dict[str, Any]]:
    """학교와 기간에 해당하는 NEIS 급식 원본 행을 가져온다."""

    validate_date_range(start, end)
    rows: list[dict[str, Any]] = []
    page = 1
    while True:
        payload = _request_json(
            "mealServiceDietInfo",
            {
                "Type": "json",
                "pIndex": page,
                "pSize": NEIS_PAGE_SIZE,
                "ATPT_OFCDC_SC_CODE": school.office_code,
                "SD_SCHUL_CODE": school.school_code,
                "MLSV_FROM_YMD": start,
                "MLSV_TO_YMD": end,
            },
            http_get=http_get,
        )
        page_rows = _extract_rows(payload, "mealServiceDietInfo")
        rows.extend(page_rows)
        total_count = _extract_total_count(payload, "mealServiceDietInfo")
        if not page_rows or total_count is None or len(rows) >= total_count:
            break
        page += 1
    return rows


## 2단계: 실시간 데이터와 예비 데이터

먼저 NEIS에 접속하고, 네트워크나 서버 문제가 생기면 노트북 안에 포함된 실제 남악고 공개 급식 5건을 사용합니다. 예비 자료를 쓰면 출처 메시지가 달라집니다.

- 예상 결과: 데이터 출처와 원본 행 수가 출력됩니다.
- 확인 질문: 예비 데이터가 없으면 수업 중 API 장애가 전체 프로젝트 중단으로 이어질 수 있습니다.


In [ ]:
EMBEDDED_SAMPLE_PAYLOAD = {'metadata': {'school_name': '남악고등학교', 'office_code': 'Q10', 'school_code': '7140272', 'source': 'NEIS 교육정보 개방 포털', 'query_start': '20260101', 'query_end': '20261231', 'fetched_at_utc': '2026-07-26T15:12:59+00:00', 'row_count': 5}, 'rows': [{'ATPT_OFCDC_SC_CODE': 'Q10', 'ATPT_OFCDC_SC_NM': '전남광주통합특별시교육청(전남)', 'SD_SCHUL_CODE': '7140272', 'SCHUL_NM': '남악고등학교', 'MMEAL_SC_CODE': '2', 'MMEAL_SC_NM': '중식', 'MLSV_YMD': '20260624', 'MLSV_FGR': 746.0, 'DDISH_NM': '양송이스프 (2.5.6.13.16)<br/>미트볼로제파스타 (1.2.5.6.10.12.13.15.16)<br/>노엣지콤비네이션피자 (1.2.5.6.10.12.13.15.16)<br/>열대과일치즈샐러드 (1.2.5.6)<br/>채소모둠피클 <br/>열무김치 (9)<br/>아이스티 (11.13)', 'ORPLC_INFO': '쇠고기(종류) : 국내산(한우)<br/>쇠고기 식육가공품 : 국내산<br/>돼지고기 : 국내산<br/>돼지고기 식육가공품 : 국내산<br/>닭고기 : 국내산<br/>닭고기 식육가공품 : 국내산<br/>오리고기 : 국내산<br/>오리고기 가공품 : 국내산<br/>쌀 : 국내산<br/>배추 : 국내산<br/>고춧가루 : 국내산<br/>콩 : 국내산<br/>콩 가공품 : 국내산<br/>낙지 : 국내산<br/>명태 : 미국산 또는 러시아<br/>고등어 : 국내산<br/>갈치 : 세네갈<br/>오징어 : 국내산<br/>꽃게 : 국내산<br/>참조기 : 국내산<br/>아귀 : 국내산<br/>주꾸미 : 베트남<br/>비고 : ', 'CAL_INFO': '1013.8 Kcal', 'NTR_INFO': '탄수화물(g) : 130.7<br/>단백질(g) : 31.0<br/>지방(g) : 41.1<br/>비타민A(R.E) : 220.8<br/>티아민(mg) : 2.1<br/>리보플라빈(mg) : 0.6<br/>비타민C(mg) : 25.3<br/>칼슘(mg) : 332.9<br/>철분(mg) : 3.1', 'MLSV_FROM_YMD': '20260624', 'MLSV_TO_YMD': '20260624', 'LOAD_DTM': '20260701'}, {'ATPT_OFCDC_SC_CODE': 'Q10', 'ATPT_OFCDC_SC_NM': '전남광주통합특별시교육청(전남)', 'SD_SCHUL_CODE': '7140272', 'SCHUL_NM': '남악고등학교', 'MMEAL_SC_CODE': '2', 'MMEAL_SC_NM': '중식', 'MLSV_YMD': '20260625', 'MLSV_FGR': 746.0, 'DDISH_NM': '친환경쌀밥 <br/>돈육애호박찌개 (10)<br/>오이부추무침 (5.6.13)<br/>묵은지닭볶음탕 (5.6.9.13.15)<br/>심쿵햄전 (1.2.5.6.10.15.16)<br/>열무김치 (9)<br/>파인애플 ', 'ORPLC_INFO': '쇠고기(종류) : 국내산(한우)<br/>쇠고기 식육가공품 : 국내산<br/>돼지고기 : 국내산<br/>돼지고기 식육가공품 : 국내산<br/>닭고기 : 국내산<br/>닭고기 식육가공품 : 국내산<br/>오리고기 : 국내산<br/>오리고기 가공품 : 국내산<br/>쌀 : 국내산<br/>배추 : 국내산<br/>고춧가루 : 국내산<br/>콩 : 국내산<br/>콩 가공품 : 국내산<br/>낙지 : 국내산<br/>명태 : 미국산 또는 러시아<br/>고등어 : 국내산<br/>갈치 : 세네갈<br/>오징어 : 국내산<br/>꽃게 : 국내산<br/>참조기 : 국내산<br/>아귀 : 국내산<br/>주꾸미 : 베트남<br/>비고 : ', 'CAL_INFO': '810.4 Kcal', 'NTR_INFO': '탄수화물(g) : 77.9<br/>단백질(g) : 49.5<br/>지방(g) : 31.1<br/>비타민A(R.E) : 166.5<br/>티아민(mg) : 0.9<br/>리보플라빈(mg) : 0.6<br/>비타민C(mg) : 33.7<br/>칼슘(mg) : 110.8<br/>철분(mg) : 3.0', 'MLSV_FROM_YMD': '20260625', 'MLSV_TO_YMD': '20260625', 'LOAD_DTM': '20260702'}, {'ATPT_OFCDC_SC_CODE': 'Q10', 'ATPT_OFCDC_SC_NM': '전남광주통합특별시교육청(전남)', 'SD_SCHUL_CODE': '7140272', 'SCHUL_NM': '남악고등학교', 'MMEAL_SC_CODE': '2', 'MMEAL_SC_NM': '중식', 'MLSV_YMD': '20260626', 'MLSV_FGR': 746.0, 'DDISH_NM': '매콤떡갈비마요덮밥 (1.5.6.10.13.15.16.18)<br/>감자호박된장국 (5.6)<br/>구운버섯샐러드 (5.6.12)<br/>바질크림츄볶이 (2.5.6.10.13.15.16)<br/>배추김치 (9)<br/>아이스구슬 (1.2.5)', 'ORPLC_INFO': '쇠고기(종류) : 국내산(한우)<br/>쇠고기 식육가공품 : 국내산<br/>돼지고기 : 국내산<br/>돼지고기 식육가공품 : 국내산<br/>닭고기 : 국내산<br/>닭고기 식육가공품 : 국내산<br/>오리고기 : 국내산<br/>오리고기 가공품 : 국내산<br/>쌀 : 국내산<br/>배추 : 국내산<br/>고춧가루 : 국내산<br/>콩 : 국내산<br/>콩 가공품 : 국내산<br/>낙지 : 국내산<br/>명태 : 미국산 또는 러시아<br/>고등어 : 국내산<br/>갈치 : 세네갈<br/>오징어 : 국내산<br/>꽃게 : 국내산<br/>참조기 : 국내산<br/>아귀 : 국내산<br/>주꾸미 : 베트남<br/>비고 : ', 'CAL_INFO': '1071.5 Kcal', 'NTR_INFO': '탄수화물(g) : 128.3<br/>단백질(g) : 29.5<br/>지방(g) : 48.5<br/>비타민A(R.E) : 155.7<br/>티아민(mg) : 1.2<br/>리보플라빈(mg) : 0.4<br/>비타민C(mg) : 9.8<br/>칼슘(mg) : 184.2<br/>철분(mg) : 3.3', 'MLSV_FROM_YMD': '20260626', 'MLSV_TO_YMD': '20260626', 'LOAD_DTM': '20260703'}, {'ATPT_OFCDC_SC_CODE': 'Q10', 'ATPT_OFCDC_SC_NM': '전남광주통합특별시교육청(전남)', 'SD_SCHUL_CODE': '7140272', 'SCHUL_NM': '남악고등학교', 'MMEAL_SC_CODE': '2', 'MMEAL_SC_NM': '중식', 'MLSV_YMD': '20260629', 'MLSV_FGR': 746.0, 'DDISH_NM': '친환경쌀밥 <br/>아욱국 (5.6)<br/>족발&찐순대 (2.5.6.10.13.16)<br/>배추겉절이 (13)<br/>무말랭이무침 <br/>비빔막국수 (3.5.6.13)<br/>피치에빠진코코 ', 'ORPLC_INFO': '쇠고기(종류) : 국내산(한우)<br/>쇠고기 식육가공품 : 국내산<br/>돼지고기 : 국내산<br/>돼지고기 식육가공품 : 국내산<br/>닭고기 : 국내산<br/>닭고기 식육가공품 : 국내산<br/>오리고기 : 국내산<br/>오리고기 가공품 : 국내산<br/>쌀 : 국내산<br/>배추 : 국내산<br/>고춧가루 : 국내산<br/>콩 : 국내산<br/>콩 가공품 : 국내산<br/>낙지 : 국내산<br/>명태 : 미국산 또는 러시아<br/>고등어 : 국내산<br/>갈치 : 세네갈<br/>오징어 : 국내산<br/>꽃게 : 국내산<br/>참조기 : 국내산<br/>아귀 : 국내산<br/>주꾸미 : 베트남<br/>비고 : ', 'CAL_INFO': '1226.5 Kcal', 'NTR_INFO': '탄수화물(g) : 228.8<br/>단백질(g) : 40.8<br/>지방(g) : 15.7<br/>비타민A(R.E) : 98.4<br/>티아민(mg) : 0.4<br/>리보플라빈(mg) : 0.5<br/>비타민C(mg) : 22.0<br/>칼슘(mg) : 389.1<br/>철분(mg) : 5.3', 'MLSV_FROM_YMD': '20260629', 'MLSV_TO_YMD': '20260629', 'LOAD_DTM': '20260706'}, {'ATPT_OFCDC_SC_CODE': 'Q10', 'ATPT_OFCDC_SC_NM': '전남광주통합특별시교육청(전남)', 'SD_SCHUL_CODE': '7140272', 'SCHUL_NM': '남악고등학교', 'MMEAL_SC_CODE': '2', 'MMEAL_SC_NM': '중식', 'MLSV_YMD': '20260630', 'MLSV_FGR': 746.0, 'DDISH_NM': '친환경잡곡밥 (5)<br/>돈육김치찌개 (5.9.10)<br/>근대나물무침 <br/>감자채햄볶음 (1.2.5.6.10.15.16)<br/>치킨까스 (1.5.6.15.18)<br/>배추김치 (9)<br/>샤인머스켓 ', 'ORPLC_INFO': '쇠고기(종류) : 국내산(한우)<br/>쇠고기 식육가공품 : 국내산<br/>돼지고기 : 국내산<br/>돼지고기 식육가공품 : 국내산<br/>닭고기 : 국내산<br/>닭고기 식육가공품 : 국내산<br/>오리고기 : 국내산<br/>오리고기 가공품 : 국내산<br/>쌀 : 국내산<br/>배추 : 국내산<br/>고춧가루 : 국내산<br/>콩 : 국내산<br/>콩 가공품 : 국내산<br/>낙지 : 국내산<br/>명태 : 미국산 또는 러시아<br/>고등어 : 국내산<br/>갈치 : 세네갈<br/>오징어 : 국내산<br/>꽃게 : 국내산<br/>참조기 : 국내산<br/>아귀 : 국내산<br/>주꾸미 : 베트남<br/>비고 : ', 'CAL_INFO': '418.0 Kcal', 'NTR_INFO': '탄수화물(g) : 72.4<br/>단백질(g) : 12.8<br/>지방(g) : 7.6<br/>비타민A(R.E) : 45.3<br/>티아민(mg) : 0.3<br/>리보플라빈(mg) : 0.2<br/>비타민C(mg) : 6.8<br/>칼슘(mg) : 53.9<br/>철분(mg) : 1.1', 'MLSV_FROM_YMD': '20260630', 'MLSV_TO_YMD': '20260630', 'LOAD_DTM': '20260707'}]}
EMBEDDED_SAMPLE_ROWS = EMBEDDED_SAMPLE_PAYLOAD['rows']


In [ ]:
SCHOOL_NAME = "남악고등학교"
QUERY_START = "20260101"
QUERY_END = "20261231"

if VERIFY_MODE:
    raw_rows = EMBEDDED_SAMPLE_ROWS
    data_source = "내장 NEIS 예비 데이터"
else:
    try:
        school = search_school(SCHOOL_NAME)
        raw_rows = fetch_meals(school, QUERY_START, QUERY_END)
        if not raw_rows:
            raise NeisApiError("선택 기간에 데이터가 없습니다.")
        data_source = "실시간 NEIS 데이터"
    except NeisApiError as error:
        raw_rows = [
            row for row in EMBEDDED_SAMPLE_ROWS
            if QUERY_START <= str(row.get("MLSV_YMD", "")) <= QUERY_END
        ]
        if not raw_rows:
            sample_dates = sorted(str(row.get("MLSV_YMD", "")) for row in EMBEDDED_SAMPLE_ROWS)
            raise NeisApiError(
                f"요청 기간과 겹치는 예비 데이터가 없습니다. "
                f"수업용 예비 데이터 기간: {sample_dates[0]}~{sample_dates[-1]}"
            ) from error
        data_source = f"내장 NEIS 예비 데이터 (사유: {error})"

print("데이터 출처:", data_source)
print("원본 급식 행:", len(raw_rows))


## 3단계: 메뉴 문자열 전처리

원본에는 `<br/>`, 알레르기 번호, `Kcal` 같은 표기가 섞여 있습니다. AI가 비교할 수 있도록 메뉴 목록과 숫자 열로 분리합니다.

- 예상 결과: 날짜, 메뉴 문장, 열량, 영양 수치가 있는 표
- 학생 도전: `split_dishes`에서 HTML 줄바꿈이 어떻게 처리되는지 찾아 표시하세요.


In [ ]:
"""NEIS 급식 원본 문자열을 학생이 분석할 수 있는 표로 바꾼다."""

from __future__ import annotations

import html
import math
import re
from datetime import datetime
from typing import Any, Iterable

import pandas as pd


ANALYSIS_COLUMNS = [
    "date",
    "school_name",
    "meal_type",
    "dishes",
    "menu_text",
    "allergy_codes",
    "calories",
    "carbs_g",
    "protein_g",
    "fat_g",
    "dish_count",
]

ALLERGY_LABELS = {
    1: "난류",
    2: "우유",
    3: "메밀",
    4: "땅콩",
    5: "대두",
    6: "밀",
    7: "고등어",
    8: "게",
    9: "새우",
    10: "돼지고기",
    11: "복숭아",
    12: "토마토",
    13: "아황산류",
    14: "호두",
    15: "닭고기",
    16: "쇠고기",
    17: "오징어",
    18: "조개류",
    19: "잣",
}

_BR_RE = re.compile(r"<br\s*/?>", flags=re.IGNORECASE)
_ALLERGY_GROUP_RE = re.compile(r"\((\d{1,2}(?:\.\d{1,2})*)\)")
_NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")


def _decoded(text: Any) -> str:
    return html.unescape(str(text or "")).strip()


def extract_allergy_codes(text: Any) -> tuple[int, ...]:
    """괄호 안 점으로 구분된 1~19 번호만 알레르기 코드로 읽는다."""

    codes: set[int] = set()
    for match in _ALLERGY_GROUP_RE.finditer(_decoded(text)):
        values = [int(value) for value in match.group(1).split(".")]
        if all(value in ALLERGY_LABELS for value in values):
            codes.update(values)
    return tuple(sorted(codes))


def split_dishes(text: Any) -> list[str]:
    """HTML 줄바꿈을 나누고 각 메뉴 끝의 알레르기 번호를 제거한다."""

    dishes: list[str] = []
    for part in _BR_RE.split(_decoded(text)):
        cleaned = _ALLERGY_GROUP_RE.sub("", part)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()
        if cleaned:
            dishes.append(cleaned)
    return dishes


def parse_calories(text: Any) -> float:
    """`927.7 Kcal` 같은 문자열에서 열량 숫자를 읽는다."""

    match = _NUMBER_RE.search(_decoded(text))
    return float(match.group()) if match else math.nan


def parse_nutrients(text: Any) -> dict[str, float]:
    """탄수화물·단백질·지방 수치를 추출하고 없으면 NaN으로 둔다."""

    decoded = _decoded(text)
    labels = {
        "carbs_g": "탄수화물",
        "protein_g": "단백질",
        "fat_g": "지방",
    }
    result: dict[str, float] = {}
    for key, korean_label in labels.items():
        match = re.search(
            rf"{korean_label}\s*\(g\)\s*:\s*(-?\d+(?:\.\d+)?)",
            decoded,
            flags=re.IGNORECASE,
        )
        result[key] = float(match.group(1)) if match else math.nan
    return result


def _date_text(value: Any) -> str | None:
    raw = str(value or "").strip()
    try:
        return datetime.strptime(raw, "%Y%m%d").date().isoformat()
    except ValueError:
        return None


def meals_to_frame(rows: Iterable[dict[str, Any]]) -> pd.DataFrame:
    """NEIS 급식 행을 고정 열을 가진 DataFrame으로 바꾼다."""

    records: list[dict[str, Any]] = []
    for row in rows:
        date_text = _date_text(row.get("MLSV_YMD"))
        dishes = split_dishes(row.get("DDISH_NM"))
        if not date_text or not dishes:
            continue
        nutrients = parse_nutrients(row.get("NTR_INFO"))
        records.append(
            {
                "date": date_text,
                "school_name": str(row.get("SCHUL_NM", "")).strip(),
                "meal_type": str(row.get("MMEAL_SC_NM", "")).strip(),
                "dishes": dishes,
                "menu_text": " ".join(dishes),
                "allergy_codes": extract_allergy_codes(row.get("DDISH_NM")),
                "calories": parse_calories(row.get("CAL_INFO")),
                **nutrients,
                "dish_count": len(dishes),
            }
        )
    return pd.DataFrame.from_records(records, columns=ANALYSIS_COLUMNS)


In [ ]:
meal_df = meals_to_frame(raw_rows)
if meal_df.empty:
    raise RuntimeError("분석할 급식 데이터가 없습니다.")
print(meal_df[["date", "menu_text", "calories", "protein_g"]].head().to_string(index=False))
print("\n분석 가능한 행:", len(meal_df))


## 4단계: 데이터 탐색과 시각화

AI를 만들기 전에 데이터의 범위와 빠진 값을 확인합니다. 그래프는 정답을 주는 장식이 아니라 데이터의 특징과 오류를 찾는 도구입니다.

- 학생 도전: 가장 열량이 높은 식단과 낮은 식단의 메뉴를 비교하세요.
- 주의: 열량이 높거나 낮다는 사실만으로 건강함을 판정하지 않습니다.


In [ ]:
summary_columns = ["calories", "carbs_g", "protein_g", "fat_g", "dish_count"]
print(meal_df[summary_columns].describe().round(1).to_string())

if importlib.util.find_spec("matplotlib") is not None and not VERIFY_MODE:
    import matplotlib.pyplot as plt
    plot_df = meal_df.sort_values("date")
    plt.figure(figsize=(10, 4))
    plt.bar(plot_df["date"], plot_df["calories"], color="#4F8BF9")
    plt.title("남악고 급식 열량 비교 — 상대 비교용")
    plt.ylabel("Kcal")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("자동 검증에서는 표로 확인했습니다. Colab에서는 막대그래프가 표시됩니다.")


## 5단계: TF-IDF와 코사인 유사도

TF-IDF는 각 메뉴에서 특징적인 글자 조각에 더 큰 값을 줍니다. 코사인 유사도는 취향 벡터와 메뉴 벡터가 같은 방향을 가리키는 정도를 0~1로 비교합니다.

이 수업에서는 라이브러리 한 줄로 숨기지 않고 문자 n-gram TF-IDF와 작은 K-Means를 직접 구현해 내부 원리를 관찰합니다.

- 확인 질문: 모든 메뉴에 자주 나오는 글자와 특정 메뉴에만 나오는 글자 중 어느 쪽이 구별에 유리할까요?


In [ ]:
"""설명 가능한 개인 취향 기반 급식 추천과 식단 군집."""

from __future__ import annotations

import math
from collections import Counter
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd


SAFETY_NOTICE = (
    "추천 결과는 취향 비교용입니다. 실제 식단과 알레르기 정보는 "
    "학교 급식표와 영양사 안내를 다시 확인하세요."
)

MENU_TYPE_KEYWORDS = {
    "밥": ("밥", "덮밥", "볶음밥", "비빔밥", "주먹밥"),
    "면": ("면", "국수", "우동", "라면", "스파게티", "파스타", "쫄면"),
    "국물": ("국", "탕", "찌개", "전골", "스프", "짬뽕"),
    "튀김": ("튀김", "돈까스", "커틀릿", "치킨", "꼬치"),
    "디저트": ("푸딩", "케이크", "과일", "바나나", "주스", "요구르트", "아이스", "우유"),
}

SPICY_KEYWORDS = {
    5: ("불닭", "마라", "매운", "핵", "아주매운"),
    4: ("짬뽕", "떡볶이", "고추", "낙지볶음"),
    3: ("김치", "제육", "비빔", "고추장"),
}


@dataclass(frozen=True)
class PreferenceProfile:
    """개인을 식별하지 않는 현재 실행용 취향 프로필."""

    likes: tuple[str, ...]
    avoids: tuple[str, ...]
    preferred_types: tuple[str, ...]
    spice_level: int
    allergy_codes: tuple[int, ...]


def _clean_terms(values: Iterable[str]) -> tuple[str, ...]:
    cleaned: list[str] = []
    for value in values:
        term = str(value).strip()
        if term and term not in cleaned:
            cleaned.append(term)
    return tuple(cleaned)


def validate_profile(profile: PreferenceProfile) -> PreferenceProfile:
    """프로필 범위를 확인하고 공백·중복을 정리한다."""

    likes = _clean_terms(profile.likes)
    avoids = _clean_terms(profile.avoids)
    preferred_types = _clean_terms(profile.preferred_types)
    allergies = tuple(sorted(set(int(code) for code in profile.allergy_codes)))
    if len(likes) > 5 or len(avoids) > 5:
        raise ValueError("좋아하거나 피하고 싶은 항목은 각각 최대 5개까지 입력하세요.")
    if not 1 <= int(profile.spice_level) <= 5:
        raise ValueError("매운맛 선호도는 1에서 5 사이여야 합니다.")
    unknown_types = set(preferred_types) - set(MENU_TYPE_KEYWORDS)
    if unknown_types:
        raise ValueError(f"지원하지 않는 메뉴 유형입니다: {', '.join(sorted(unknown_types))}")
    if any(code < 1 or code > 19 for code in allergies):
        raise ValueError("알레르기 주의 번호는 1에서 19 사이여야 합니다.")
    return PreferenceProfile(likes, avoids, preferred_types, int(profile.spice_level), allergies)


def _char_ngrams(text: str) -> Counter[str]:
    counts: Counter[str] = Counter()
    for word in text.casefold().split():
        padded = f" {word} "
        for size in (2, 3, 4):
            counts.update(padded[index : index + size] for index in range(len(padded) - size + 1))
    return counts


def _tfidf_similarity(menu_texts: list[str], query: str) -> np.ndarray:
    """작은 수업 데이터에 맞춘 문자 n-gram TF-IDF 코사인 유사도."""

    if not query.strip():
        return np.zeros(len(menu_texts), dtype=float)
    documents = [_char_ngrams(text) for text in [*menu_texts, query]]
    document_count = len(documents)
    document_frequency: Counter[str] = Counter()
    for counts in documents:
        document_frequency.update(counts.keys())
    idf = {
        term: math.log((1 + document_count) / (1 + frequency)) + 1.0
        for term, frequency in document_frequency.items()
    }

    def vector(counts: Counter[str]) -> dict[str, float]:
        total = sum(counts.values()) or 1
        return {term: (count / total) * idf[term] for term, count in counts.items()}

    vectors = [vector(counts) for counts in documents]
    query_vector = vectors[-1]
    query_norm = math.sqrt(sum(value * value for value in query_vector.values())) or 1.0
    similarities: list[float] = []
    for menu_vector in vectors[:-1]:
        dot = sum(value * query_vector.get(term, 0.0) for term, value in menu_vector.items())
        menu_norm = math.sqrt(sum(value * value for value in menu_vector.values())) or 1.0
        similarities.append(dot / (menu_norm * query_norm))
    return np.asarray(similarities, dtype=float)


def _menu_types(menu_text: str) -> tuple[str, ...]:
    lowered = menu_text.casefold()
    return tuple(
        menu_type
        for menu_type, keywords in MENU_TYPE_KEYWORDS.items()
        if any(keyword.casefold() in lowered for keyword in keywords)
    )


def _spice_level(menu_text: str) -> int:
    lowered = menu_text.casefold()
    for level in (5, 4, 3):
        if any(keyword.casefold() in lowered for keyword in SPICY_KEYWORDS[level]):
            return level
    return 2


def cluster_meals(frame: pd.DataFrame, max_clusters: int = 3) -> pd.DataFrame:
    """영양 수치가 비슷한 식단을 작은 결정적 K-Means로 묶는다."""

    result = frame.copy()
    if len(result) < 3:
        result["cluster_name"] = "데이터 부족"
        return result

    columns = ["calories", "carbs_g", "protein_g", "fat_g", "dish_count"]
    features = result[columns].apply(pd.to_numeric, errors="coerce")
    nutrition_columns = ["calories", "carbs_g", "protein_g", "fat_g"]
    complete_nutrition_rows = int(features[nutrition_columns].notna().all(axis=1).sum())
    if complete_nutrition_rows < 3:
        result["cluster_name"] = "데이터 부족"
        return result
    features = features.fillna(features.median(numeric_only=True)).fillna(0.0)
    matrix = features.to_numpy(dtype=float)
    means = matrix.mean(axis=0)
    stds = matrix.std(axis=0)
    stds[stds == 0] = 1.0
    standardized = (matrix - means) / stds

    unique_count = len(np.unique(standardized, axis=0))
    cluster_count = min(max(1, int(max_clusters)), len(result), unique_count)
    if cluster_count == 1:
        result["cluster_name"] = "중간 구성"
        return result

    order = np.argsort(features["calories"].to_numpy())
    positions = np.linspace(0, len(order) - 1, cluster_count).round().astype(int)
    centroids = standardized[order[positions]].copy()
    labels = np.zeros(len(result), dtype=int)
    for _ in range(30):
        distances = ((standardized[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
        new_labels = distances.argmin(axis=1)
        new_centroids = centroids.copy()
        for cluster_id in range(cluster_count):
            members = standardized[new_labels == cluster_id]
            if len(members):
                new_centroids[cluster_id] = members.mean(axis=0)
        if np.array_equal(new_labels, labels) and np.allclose(new_centroids, centroids):
            labels = new_labels
            break
        labels, centroids = new_labels, new_centroids

    calorie_values = features["calories"].to_numpy(dtype=float)
    calorie_means = {
        cluster_id: float(calorie_values[labels == cluster_id].mean())
        for cluster_id in range(cluster_count)
        if np.any(labels == cluster_id)
    }
    sorted_clusters = sorted(calorie_means, key=calorie_means.get)
    if len(sorted_clusters) == 2:
        names = {
            sorted_clusters[0]: "상대적 가벼운 구성",
            sorted_clusters[1]: "상대적 든든한 구성",
        }
    else:
        names = {
            sorted_clusters[0]: "상대적 가벼운 구성",
            sorted_clusters[-1]: "상대적 든든한 구성",
            **{cluster_id: "중간 구성" for cluster_id in sorted_clusters[1:-1]},
        }
    result["cluster_name"] = [names[int(label)] for label in labels]
    return result


def _empty_result(frame: pd.DataFrame, excluded_count: int) -> pd.DataFrame:
    result = frame.iloc[0:0].copy()
    for column in ("score", "reason", "cluster_name", "safety_notice"):
        if column not in result:
            result[column] = pd.Series(dtype="object")
    result.attrs["excluded_count"] = excluded_count
    return result


def recommend_menus(
    frame: pd.DataFrame,
    profile: PreferenceProfile,
    top_n: int = 3,
) -> pd.DataFrame:
    """취향 유사도와 명시적 가감점을 합쳐 추천 순위를 만든다."""

    profile = validate_profile(profile)
    if top_n < 1:
        raise ValueError("추천 개수는 1개 이상이어야 합니다.")
    required = {"menu_text", "allergy_codes", "date"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"급식 데이터에 필요한 열이 없습니다: {', '.join(sorted(missing))}")

    allergy_set = set(profile.allergy_codes)
    unsafe_mask = frame["allergy_codes"].apply(lambda codes: bool(allergy_set.intersection(set(codes))))
    excluded_count = int(unsafe_mask.sum())
    safe = frame.loc[~unsafe_mask].copy().reset_index(drop=True)
    if safe.empty:
        return _empty_result(frame, excluded_count)

    safe = cluster_meals(safe)
    query_parts = [*profile.likes, *profile.likes, *profile.preferred_types]
    query = " ".join(query_parts)
    similarities = _tfidf_similarity(safe["menu_text"].astype(str).tolist(), query)

    scores: list[float] = []
    reasons: list[str] = []
    for position, row in safe.iterrows():
        menu_text = str(row["menu_text"])
        lowered = menu_text.casefold()
        like_hits = [term for term in profile.likes if term.casefold() in lowered]
        avoid_hits = [term for term in profile.avoids if term.casefold() in lowered]
        menu_types = _menu_types(menu_text)
        type_hits = [menu_type for menu_type in profile.preferred_types if menu_type in menu_types]
        spice_difference = abs(profile.spice_level - _spice_level(menu_text))

        # 20점 기준점은 기피·매운맛 감점이 0점 하한에서도 보이게 하기 위한 공개 상수다.
        score = 20.0 + similarities[position] * 70.0
        score += 8.0 * len(like_hits)
        score += 5.0 * len(type_hits)
        score -= 18.0 * len(avoid_hits)
        score -= 3.0 * spice_difference
        scores.append(round(float(np.clip(score, 0.0, 100.0)), 1))

        reason_parts = [f"텍스트 유사도 {similarities[position]:.2f}"]
        if like_hits:
            reason_parts.append(f"좋아하는 키워드: {', '.join(like_hits)}")
        if type_hits:
            reason_parts.append(f"선호 유형: {', '.join(type_hits)}")
        if avoid_hits:
            reason_parts.append(f"피하고 싶은 키워드: {', '.join(avoid_hits)}")
        reason_parts.append(f"매운맛 차이 {spice_difference}")
        reasons.append(" · ".join(reason_parts))

    safe["score"] = scores
    safe["reason"] = reasons
    safe["safety_notice"] = SAFETY_NOTICE
    result = safe.sort_values(["score", "date"], ascending=[False, True]).head(top_n).reset_index(drop=True)
    result.attrs["excluded_count"] = excluded_count
    return result


## 6단계: K-Means로 식단 유형 찾기

K-Means는 정답표 없이 비슷한 숫자 패턴을 가까운 중심점에 묶습니다. 열량·탄수화물·단백질·지방·메뉴 수를 사용합니다.

군집명은 데이터 안에서의 상대적 차이일 뿐 `건강함`이나 `좋음`을 뜻하지 않습니다.


In [ ]:
clustered_df = cluster_meals(meal_df, max_clusters=3)
print(clustered_df[["date", "calories", "protein_g", "cluster_name"]].to_string(index=False))


## 7단계: 익명 개인 취향 프로필 만들기

아래 값은 알고리즘 시험용 **가상 학생 프로필**입니다. 이름이나 학번은 만들지 않습니다. 좋아하는 메뉴를 바꾸어 실험할 수 있지만, 실제 알레르기 정보는 코드 셀에 적지 말고 `allergy_codes=()`를 유지하세요.

- `likes`: 좋아하는 재료·메뉴 최대 5개
- `avoids`: 피하고 싶은 재료·메뉴 최대 5개
- `preferred_types`: 밥, 면, 국물, 튀김, 디저트
- `spice_level`: 1~5
- `allergy_codes`: NEIS 알레르기 주의 번호 1~19, 선택 입력


In [ ]:
demo_profile = PreferenceProfile(
    likes=("파스타", "피자"),
    avoids=("오이",),
    preferred_types=("면", "디저트"),
    spice_level=2,
    allergy_codes=(),
)
validate_profile(demo_profile)
print("가상 취향 프로필 준비 완료 (개인 식별 정보·실제 알레르기 입력 없음)")


## 8단계: 개인별 추천 결과와 이유

점수는 아래 수식을 0~100점으로 자른 상대 점수입니다. 20점 기준점은 감점이 0점 아래로 즉시 사라지지 않게 합니다.

`20 + 70×텍스트유사도 + 8×좋아하는키워드수 + 5×선호유형수 - 18×기피키워드수 - 3×매운맛차이`

알레르기 주의 번호가 겹치면 점수 계산 전에 추천 후보에서 제외합니다.

- 학생 도전: 취향을 바꾸고 1위가 왜 달라졌는지 `추천 이유`로 설명하세요.


In [ ]:
"""실시간 NEIS 조회, 예비 데이터, 추천 결과를 한 흐름으로 연결한다."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Callable, Iterable

import pandas as pd



Fetcher = Callable[[str, str, str], list[dict]]
DEFAULT_SCHOOL = "남악고등학교"


def _live_fetcher(school_name: str, start: str, end: str) -> list[dict]:
    school = search_school(school_name)
    return fetch_meals(school, start, end)


def _load_fallback(path: Path, school_name: str) -> list[dict]:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        raise NeisApiError("예비 급식 데이터도 읽지 못했습니다.") from exc
    metadata = payload.get("metadata", {})
    rows = payload.get("rows")
    if metadata.get("school_name") != school_name or not isinstance(rows, list):
        raise NeisApiError("예비 급식 데이터의 학교명 또는 구조가 올바르지 않습니다.")
    return rows


def _filter_rows_by_date(rows: list[dict], start: str, end: str) -> list[dict]:
    return [row for row in rows if start <= str(row.get("MLSV_YMD", "")) <= end]


def load_meal_frame(
    school_name: str,
    start: str,
    end: str,
    fallback_path: str | Path,
    *,
    fetcher: Fetcher | None = None,
) -> tuple[pd.DataFrame, str]:
    """실시간 급식을 우선 사용하고 연결 실패 때만 남악고 예비 데이터로 전환한다."""

    validate_date_range(start, end)
    cleaned_school = school_name.strip()
    if not cleaned_school:
        raise ValueError("학교명을 입력하세요.")
    selected_fetcher = fetcher or _live_fetcher
    try:
        rows = selected_fetcher(cleaned_school, start, end)
    except NeisApiError as exc:
        if cleaned_school != DEFAULT_SCHOOL:
            raise
        fallback_rows = _load_fallback(Path(fallback_path), cleaned_school)
        rows = _filter_rows_by_date(fallback_rows, start, end)
        if not rows:
            available_dates = sorted(
                str(row.get("MLSV_YMD", ""))
                for row in fallback_rows
                if str(row.get("MLSV_YMD", ""))
            )
            period = (
                f"{available_dates[0]}~{available_dates[-1]}"
                if available_dates
                else "확인할 수 없음"
            )
            raise NeisApiError(
                f"실시간 조회에 실패했고 요청 기간({start}~{end})과 겹치는 예비 데이터가 없습니다. "
                f"예비 데이터 기간: {period}. 수업용 시연은 이 기간으로 조회하세요."
            ) from exc
        frame = meals_to_frame(rows)
        if frame.empty:
            raise NeisApiError("예비 급식 데이터가 비어 있습니다.") from exc
        return frame, f"남악고 예비 데이터 사용 · 실시간 조회 사유: {exc}"

    frame = meals_to_frame(rows)
    if frame.empty:
        raise ValueError("선택한 기간에 급식 데이터가 없습니다. 다른 기간을 선택하세요.")
    return frame, "실시간 NEIS 데이터 사용"


def _csv_terms(text: str) -> tuple[str, ...]:
    return tuple(part.strip() for part in str(text or "").split(",") if part.strip())


def run_recommendation(
    frame: pd.DataFrame,
    *,
    likes_text: str,
    avoids_text: str,
    preferred_types: Iterable[str],
    spice_level: int,
    allergy_codes: Iterable[int],
    top_n: int = 3,
) -> tuple[str, pd.DataFrame]:
    """화면 입력을 익명 프로필로 바꾸고 한국어 결과 표를 만든다."""

    profile = PreferenceProfile(
        likes=_csv_terms(likes_text),
        avoids=_csv_terms(avoids_text),
        preferred_types=tuple(preferred_types or ()),
        spice_level=int(spice_level),
        allergy_codes=tuple(int(code) for code in (allergy_codes or ())),
    )
    result = recommend_menus(frame, profile, top_n=top_n)
    excluded_count = int(result.attrs.get("excluded_count", 0))
    if result.empty:
        summary = (
            f"선택한 알레르기 주의 번호 때문에 모든 메뉴가 제외되었습니다. "
            f"제외된 메뉴: {excluded_count}개\n\n{SAFETY_NOTICE}"
        )
        return summary, pd.DataFrame(
            columns=["순위", "날짜", "추천 점수", "메뉴", "추천 이유", "식단 군집", "알레르기 번호"]
        )

    table = pd.DataFrame(
        {
            "순위": range(1, len(result) + 1),
            "날짜": result["date"],
            "추천 점수": result["score"],
            "메뉴": result["menu_text"],
            "추천 이유": result["reason"],
            "식단 군집": result["cluster_name"],
            "알레르기 번호": result["allergy_codes"].apply(
                lambda codes: ", ".join(str(code) for code in codes) if codes else "없음"
            ),
        }
    )
    summary = (
        f"취향을 비교해 {len(table)}개 메뉴를 추천했습니다. "
        f"알레르기 주의 번호로 제외된 메뉴는 {excluded_count}개입니다.\n\n{SAFETY_NOTICE}"
    )
    return summary, table


In [ ]:
recommendation_summary, recommendation_result = run_recommendation(
    meal_df,
    likes_text=", ".join(demo_profile.likes),
    avoids_text=", ".join(demo_profile.avoids),
    preferred_types=demo_profile.preferred_types,
    spice_level=demo_profile.spice_level,
    allergy_codes=demo_profile.allergy_codes,
    top_n=3,
)
print(recommendation_summary)
print(recommendation_result.to_string(index=False))


## 9단계: 추천 결과 시험하기

AI 서비스는 한 번 실행되는 것보다 입력을 바꿔도 규칙대로 움직이는지가 중요합니다. 서로 다른 두 취향의 1위를 비교합니다.

- 시험 A: 면·피자 선호
- 시험 B: 밥·국물 선호
- 성공 기준: 두 결과의 이유를 데이터로 설명할 수 있다.


In [ ]:
profile_a = PreferenceProfile(("파스타", "피자"), (), ("면",), 2, ())
profile_b = PreferenceProfile(("밥", "국"), (), ("밥", "국물"), 3, ())
result_a = recommend_menus(meal_df, profile_a, top_n=1)
result_b = recommend_menus(meal_df, profile_b, top_n=1)
print("A 프로필 1위:", result_a.iloc[0]["menu_text"], "/", result_a.iloc[0]["reason"])
print("B 프로필 1위:", result_b.iloc[0]["menu_text"], "/", result_b.iloc[0]["reason"])


In [ ]:
"""Gradio 실행 환경에 따른 공개 범위를 명시적으로 결정한다."""

from __future__ import annotations

import sys


def is_google_colab() -> bool:
    return "google.colab" in sys.modules


def launch_options(*, is_colab: bool) -> dict[str, bool]:
    """Colab에서는 접속 가능한 임시 링크를, 로컬에서는 localhost를 사용한다."""

    return {
        "share": bool(is_colab),
        "debug": False,
        "show_error": True,
    }


## 10단계: Gradio 서비스 화면

아래 셀을 실행하면 입력 상자와 추천 버튼이 있는 서비스가 열립니다. Colab에서는 접속 가능한 임시 공개 링크가 만들어집니다. 이름·학번은 입력하지 말고, 알레르기 번호도 실제 정보 대신 발표용 가상 번호로만 시험합니다.

발표에서는 `입력 → 데이터 → AI 비교 → 추천 결과 → 한계` 순서로 시연하세요.


In [ ]:
def build_colab_demo(current_frame):
    import gradio as gr

    def recommend_ui(likes, avoids, menu_types, spice, allergies):
        summary, table = run_recommendation(
            current_frame,
            likes_text=likes,
            avoids_text=avoids,
            preferred_types=menu_types,
            spice_level=int(spice),
            allergy_codes=[int(code) for code in allergies],
        )
        return summary, table

    with gr.Blocks(title="우리 학교 급식 AI 개인추천기") as demo:
        gr.Markdown(
            "# 🍱 우리 학교 급식 AI 개인추천기\n"
            "이름·학번은 입력하지 않습니다. Colab 링크는 임시 공개 링크입니다.\n\n"
            "⚠️ 실제 알레르기·질병 정보는 입력하지 말고 수업용 가상 번호만 사용하세요."
        )
        likes = gr.Textbox(label="좋아하는 재료·메뉴", value="파스타, 피자")
        avoids = gr.Textbox(label="피하고 싶은 재료·메뉴", value="오이")
        menu_types = gr.CheckboxGroup(["밥", "면", "국물", "튀김", "디저트"], label="선호 유형")
        spice = gr.Slider(1, 5, value=3, step=1, label="매운맛 선호도")
        allergies = gr.CheckboxGroup(
            [str(code) for code in range(1, 20)],
            label="알레르기 주의 번호(가상 시연 전용)",
        )
        button = gr.Button("나에게 맞는 급식 찾기", variant="primary")
        message = gr.Markdown()
        table = gr.Dataframe(interactive=False)
        button.click(recommend_ui, [likes, avoids, menu_types, spice, allergies], [message, table])
        gr.Markdown("⚠️ 실제 식단과 알레르기 정보는 학교 급식표와 영양사 안내를 다시 확인하세요.")
    return demo

if VERIFY_MODE:
    print("Gradio 화면 함수 정의 완료 - 자동 검증에서는 서버를 열지 않습니다.")
else:
    demo = build_colab_demo(meal_df)
    demo.launch(**launch_options(is_colab=is_google_colab()))


## 11단계: 모델 카드와 발표 준비

모델 카드는 AI가 무엇을 하고, 어떤 데이터를 쓰며, 어디에서 틀릴 수 있는지 공개하는 설명서입니다.

### 반드시 말할 한계

- 추천 점수는 취향 유사도이지 건강 점수가 아니다.
- 메뉴명에 없는 재료는 AI가 알 수 없다.
- 알레르기 번호 누락이나 식단 변경 가능성이 있다.
- 학생 6명의 취향이 모든 학생을 대표하지 않는다.
- 생성형 AI로 발표문을 만들었다면 NEIS 원본과 다시 비교한다.


In [ ]:
model_card = {
    "서비스 목적": "익명 취향과 NEIS 메뉴 텍스트를 비교한 상대 추천",
    "사용 데이터": data_source,
    "AI 방법": "문자 n-gram TF-IDF, 코사인 유사도, 작은 K-Means",
    "수집하지 않는 정보": "이름, 학번, 반, 연락처, 체중, 질병명",
    "안전 문구": SAFETY_NOTICE,
    "핵심 한계": "메뉴명과 선택한 취향만 비교하므로 실제 만족도나 건강 적합도를 예측하지 않음",
}
for key, value in model_card.items():
    print(f"- {key}: {value}")
